In [ ]:
from datetime import datetime
from pathlib import Path

import polars as pl
import polars_distance as pld

In [2]:
DIR = "/mnt/common-ssd/group-dirs/tkom-2026-project/pickwell/budapest/partitioned/"

In [11]:
test = pl.DataFrame(
    {
        "device_id": [1, 1, 1, 1, 2, 2, 2],
        "timestamp": [
            datetime(2026, 10, 19, 10, 42, 38),
            datetime(2026, 10, 19, 10, 42, 59),
            datetime(2026, 10, 19, 10, 44, 11),
            datetime(2026, 10, 19, 10, 43, 7),
            datetime(2026, 10, 18, 17, 15, 43),
            datetime(2026, 10, 18, 17, 15, 49),
            datetime(2026, 10, 18, 17, 18, 2),
        ],
        "longitude": [
            19.033973,
            19.0334969,
            19.0321977,
            19.0326872,
            19.1095298,
            19.1099686,
            19.1165223,
        ],
        "latitude": [
            47.501812,
            47.501803,
            47.501977,
            47.501878,
            47.472143,
            47.471786,
            47.470600,
        ],
    }
)

In [9]:
def calculate_distance(data: pl.DataFrame) -> pl.DataFrame:
    return (
        data.with_columns(
            pl.col("latitude")
            .shift()
            .over("device_id", order_by="timestamp")
            .alias("prev_latitude"),
            pl.col("longitude")
            .shift()
            .over("device_id", order_by="timestamp")
            .alias("prev_longitude"),
        )
        .with_columns(
            pl.struct(
                pl.col("latitude").alias("latitude"),
                pl.col("longitude").alias("longitude"),
            ).alias("current_point"),
            pl.struct(
                pl.col("prev_latitude").alias("latitude"),
                pl.col("prev_longitude").alias("longitude"),
            ).alias("previous_point"),
        )
        .with_columns(
            pld.col("current_point")
            .dist.haversine("previous_point", unit="km")
            .alias("distance_km")
        )
        .with_columns(
            pl.col("timestamp")
            .diff()
            .over("device_id", order_by="timestamp")
            .alias("time_diff")
        )
    )

In [14]:
calculate_distance(test.sort(["device_id", "timestamp"]))

device_id,timestamp,longitude,latitude,prev_latitude,prev_longitude,current_point,previous_point,distance_km,time_diff
i64,datetime[μs],f64,f64,f64,f64,struct[2],struct[2],f64,duration[μs]
1,2026-10-19 10:42:38,19.033973,47.501812,null,null,"{47.501812,19.033973}","{null,null}",null,null
1,2026-10-19 10:42:59,19.033497,47.501803,47.501812,19.033973,"{47.501803,19.033497}","{47.501812,19.033973}",0.035778,21s
1,2026-10-19 10:43:07,19.032687,47.501878,47.501803,19.033497,"{47.501878,19.032687}","{47.501803,19.033497}",0.061393,8s
1,2026-10-19 10:44:11,19.032198,47.501977,47.501878,19.032687,"{47.501977,19.032198}","{47.501878,19.032687}",0.038383,1m 4s
2,2026-10-18 17:15:43,19.10953,47.472143,null,null,"{47.472143,19.10953}","{null,null}",null,null
2,2026-10-18 17:15:49,19.109969,47.471786,47.472143,19.10953,"{47.471786,19.109969}","{47.472143,19.10953}",0.05161,6s
2,2026-10-18 17:18:02,19.116522,47.4706,47.471786,19.109969,"{47.4706,19.116522}","{47.471786,19.109969}",0.509946,2m 13s


In [ ]:
Path(f"{DIR}/with_distance/").mkdir(exist_ok=True, parents=True)
for i in range(2):
    df = pl.read_parquet(f"{DIR}/{i:02x}/pings.parquet")
    df = calculate_distance(df)
    df.drop(
        ["prev_latitude", "prev_longitude", "current_point", "previous_point"]
    ).write_parquet(f"{DIR}/with_distance/{i:02x}.parquet")

In [ ]:
df.drop(["prev_latitude", "prev_longitude", "current_point", "previous_point"])

timestamp,device_id,device_aid_type,latitude,longitude,id_with_type,distance_km,time_diff
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,str,f64,duration[μs]
2025-08-30 06:25:28 CEST,6587,"""AAID""",47.528802,19.049234,"""0100028f0f10a6257e84c46662f199…",null,null
2024-01-17 00:58:13 CET,6588,"""AAID""",47.556306,19.102451,"""01000917fe3ff163cfe38d480258c4…",null,null
2024-01-17 01:17:04 CET,6588,"""AAID""",47.556303,19.102439,"""01000917fe3ff163cfe38d480258c4…",0.000991,18m 51s
2024-01-17 01:18:13 CET,6588,"""AAID""",47.556306,19.102451,"""01000917fe3ff163cfe38d480258c4…",0.000991,1m 9s
2024-01-17 01:37:35 CET,6588,"""AAID""",47.556306,19.102451,"""01000917fe3ff163cfe38d480258c4…",0.0,19m 22s
…,…,…,…,…,…,…,…
2025-12-21 15:19:34 CET,13220,"""AAID""",47.663946,19.609275,"""01fff1da229019692a1ed0ad2cddd1…",10.652959,5m 12s
2025-12-21 15:25:04 CET,13220,"""AAID""",47.684819,19.668607,"""01fff1da229019692a1ed0ad2cddd1…",5.012059,5m 30s
2025-12-21 15:40:04 CET,13220,"""AAID""",47.706548,20.110039,"""01fff1da229019692a1ed0ad2cddd1…",33.1258,15m
